# HowLongToBeat — Exploratory Data Analysis

Analysis of 10,000+ game completion time entries scraped from howlongtobeat.com.
This notebook explores the distribution of completion times, genre patterns, and data quality.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------- Design System ----------
PALETTE = ["#2ecc71", "#3498db", "#e74c3c", "#f39c12", "#9b59b6",
           "#1abc9c", "#e67e22", "#34495e", "#d35400", "#8e44ad"]
sns.set_theme(style="whitegrid", palette=PALETTE,
              rc={"figure.dpi": 120, "savefig.dpi": 150,
                  "font.family": "sans-serif"})
plt.rcParams.update({"axes.titlesize": 14, "axes.labelsize": 11})

DATA_DIR = "../data"

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/hltb_games.csv", encoding="utf-8-sig")
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 1 — Missing Values Analysis

In [ ]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.4]})

# Bar chart
colors = ["#e74c3c" if v > 50 else "#f39c12" if v > 10 else "#2ecc71" for v in missing_pct]
missing_pct.plot.barh(ax=axes[0], color=colors)
axes[0].set_xlabel("Missing (%)")
axes[0].set_title("Missing Values by Column")
for i, v in enumerate(missing_pct):
    axes[0].text(v + 0.5, i, f"{v:.1f}%", va="center", fontsize=8)

# Matrix heatmap
sample = df.sample(min(200, len(df)), random_state=42).sort_index()
sns.heatmap(sample.isnull(), cbar=False, yticklabels=False, ax=axes[1],
            cmap=["#2ecc71", "#ecf0f1"])
axes[1].set_title("Missing Pattern (200-row sample)")
axes[1].set_ylabel("Row")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/missing_values.png", bbox_inches="tight")
plt.show()

The missing values chart shows data completeness across all columns. Fields like `genre` and `developer` have higher missing rates because they are only available through Selenium detail page enrichment, which is applied to a sample of listings.

## 2 — Main Story Completion Time Distribution

In [ ]:
main_story = df["main_story_hours"].dropna()
main_story = main_story[main_story > 0]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(main_story, bins=100, color=PALETTE[0], alpha=0.8, edgecolor="white")
ax.set_xscale("log")
ax.set_xlabel("Main Story Time (hours, log scale)")
ax.set_ylabel("Count")
ax.set_title("Distribution of Main Story Completion Times")

median_val = main_story.median()
mean_val = main_story.mean()
ax.axvline(median_val, color="#e74c3c", linestyle="--", linewidth=2,
           label=f"Median: {median_val:.1f}h")
ax.axvline(mean_val, color="#3498db", linestyle="-.", linewidth=2,
           label=f"Mean: {mean_val:.1f}h")
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/main_story_distribution.png", bbox_inches="tight")
plt.show()

The histogram reveals a strongly right-skewed distribution of main story completion times, which is why a logarithmic x-axis is necessary. The gap between the mean and median indicates that a small number of extremely long games (100+ hours) pull the average significantly above the typical experience.

## 3 — Completion Time Categories Comparison

In [ ]:
time_cols = ["main_story_hours", "main_plus_extras_hours", "completionist_hours"]
time_data = df[time_cols].dropna(how="all")
time_melted = time_data.melt(var_name="Category", value_name="Hours")
time_melted = time_melted.dropna()
time_melted = time_melted[time_melted["Hours"] > 0]

label_map = {
    "main_story_hours": "Main Story",
    "main_plus_extras_hours": "Main + Extras",
    "completionist_hours": "Completionist",
}
time_melted["Category"] = time_melted["Category"].map(label_map)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=time_melted, x="Category", y="Hours", palette=PALETTE[:3], ax=ax,
            showfliers=False)
ax.set_title("Completion Times: Main Story vs. Extras vs. Completionist")
ax.set_ylabel("Hours")
ax.set_xlabel("")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/time_categories_boxplot.png", bbox_inches="tight")
plt.show()

The box plot reveals the substantial additional content beyond the critical path in most games — completionist times are typically 2–3× longer than main story times, showing that side content and collectibles often exceed the core narrative in total playtime.

## 4 — Main Story vs. Completionist Scatter

In [ ]:
scatter_df = df[["main_story_hours", "completionist_hours", "platform"]].dropna()
scatter_df = scatter_df[
    (scatter_df["main_story_hours"] > 0)
    & (scatter_df["completionist_hours"] > 0)
]

# Use the first platform as a rough genre proxy if genre is mostly missing
scatter_df["primary_platform"] = scatter_df["platform"].str.split(",").str[0].str.strip()
top_platforms = scatter_df["primary_platform"].value_counts().head(8).index
scatter_df["platform_group"] = scatter_df["primary_platform"].where(
    scatter_df["primary_platform"].isin(top_platforms), "Other"
)

fig, ax = plt.subplots(figsize=(12, 8))
for i, (name, group) in enumerate(scatter_df.groupby("platform_group")):
    ax.scatter(group["main_story_hours"], group["completionist_hours"],
               alpha=0.4, s=15, label=name, color=PALETTE[i % len(PALETTE)])

# Diagonal reference line (y=x means no extra content)
max_val = max(scatter_df["main_story_hours"].max(), scatter_df["completionist_hours"].max())
ax.plot([0, max_val], [0, max_val], "--", color="gray", alpha=0.5, label="y = x (no extras)")

ax.set_xlabel("Main Story (hours)")
ax.set_ylabel("Completionist (hours)")
ax.set_title("Main Story vs. Completionist Time (by Platform)")
ax.legend(fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/story_vs_completionist.png", bbox_inches="tight")
plt.show()

The scatter plot shows that most games cluster along a consistent ratio between main story and completionist time, but a distinct group of titles — particularly on PC — show exceptionally high completionist-to-main-story ratios, suggesting extensive optional content or collectible systems.

## 5 — Top 20 Platforms by Median Main Story Time

In [ ]:
# Explode multi-platform entries into individual rows
platform_df = df[["main_story_hours", "platform"]].dropna()
platform_df = platform_df[platform_df["main_story_hours"] > 0]
platform_df["platform_list"] = platform_df["platform"].str.split(", ")
platform_exploded = platform_df.explode("platform_list")

platform_median = (
    platform_exploded.groupby("platform_list")["main_story_hours"]
    .agg(["median", "count"])
    .query("count >= 10")
    .sort_values("median", ascending=True)
    .tail(20)
)

fig, ax = plt.subplots(figsize=(10, 8))
platform_median["median"].plot.barh(ax=ax, color=PALETTE[1])
ax.set_xlabel("Median Main Story Time (hours)")
ax.set_title("Top 20 Platforms by Median Main Story Time")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/platform_median_time.png", bbox_inches="tight")
plt.show()

This chart reveals which platforms host the longest games on average, directly answering which gaming ecosystems demand the most time investment from players.

## 6 — Game Releases Per Year

In [ ]:
year_df = df["release_year"].dropna().astype(int)
year_df = year_df[(year_df >= 1980) & (year_df <= 2025)]
year_counts = year_df.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(year_counts.index, year_counts.values, color=PALETTE[0], alpha=0.7, width=0.8)

# Rolling average
rolling = year_counts.rolling(window=3, center=True).mean()
ax.plot(rolling.index, rolling.values, color=PALETTE[2], linewidth=2.5,
        label="3-year rolling average")

ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Games")
ax.set_title("Game Releases per Year in HLTB Database")
ax.legend()

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/releases_per_year.png", bbox_inches="tight")
plt.show()

The time series shows a sharp increase in HLTB database entries from the mid-2000s onwards, reflecting both the growth of the indie game market and the increasing adoption of HowLongToBeat as a community tracking tool.

## 7 — Correlation Heatmap

In [ ]:
numeric_cols = ["main_story_hours", "main_plus_extras_hours", "completionist_hours",
                "all_styles_hours", "review_score", "count_comp", "count_review"]
corr_df = df[numeric_cols].dropna(how="all")

fig, ax = plt.subplots(figsize=(9, 7))
corr_matrix = corr_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            mask=mask, square=True, ax=ax, linewidths=0.5,
            annot_kws={"fontsize": 10})
ax.set_title("Correlation Matrix — Numeric Fields")

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/correlation_heatmap.png", bbox_inches="tight")
plt.show()

The three completion time categories (main, extras, completionist) are strongly positively correlated, confirming that longer games tend to be longer across all play styles. Review scores show weak-to-no correlation with playtime, suggesting that game length is not a significant predictor of quality.

## 8 — Completionist-to-Main-Story Ratio by Platform

In [ ]:
ratio_df = df[["main_story_hours", "completionist_hours", "platform"]].dropna()
ratio_df = ratio_df[(ratio_df["main_story_hours"] > 0.5) & (ratio_df["completionist_hours"] > 0)]
ratio_df["comp_ratio"] = ratio_df["completionist_hours"] / ratio_df["main_story_hours"]

# Explode platforms
ratio_df["platform_list"] = ratio_df["platform"].str.split(", ")
ratio_exploded = ratio_df.explode("platform_list")

platform_ratio = (
    ratio_exploded.groupby("platform_list")["comp_ratio"]
    .agg(["median", "count"])
    .query("count >= 10")
    .sort_values("median", ascending=True)
    .tail(15)
)

fig, ax = plt.subplots(figsize=(10, 7))
platform_ratio["median"].plot.barh(ax=ax, color=PALETTE[4])
ax.set_xlabel("Median Completionist / Main Story Ratio")
ax.set_title("Which Platforms Reward Completionists Most?")
ax.axvline(x=1.0, color="gray", linestyle="--", alpha=0.5, label="1:1 ratio")
ax.legend()

plt.tight_layout()
plt.savefig(f"{DATA_DIR}/completionist_ratio.png", bbox_inches="tight")
plt.show()

This analysis identifies which gaming platforms have the highest completionist-to-main-story ratio, revealing where optional content and collectibles add the most relative playtime beyond the core experience.

## Key Findings

In [ ]:
# Compute key statistics for the findings section
total_games = len(df)
ms = df["main_story_hours"].dropna()
ms_pos = ms[ms > 0]
ms_median = ms_pos.median()
ms_mean = ms_pos.mean()

comp = df["completionist_hours"].dropna()
comp_pos = comp[comp > 0]

# Overall completionist ratio
both = df[["main_story_hours", "completionist_hours"]].dropna()
both = both[(both["main_story_hours"] > 0.5) & (both["completionist_hours"] > 0)]
overall_ratio = (both["completionist_hours"] / both["main_story_hours"]).median()

# Data quality
has_main = df["main_story_hours"].notna().sum()
pct_main = has_main / total_games * 100

# Review score vs playtime correlation
corr_review = df[["main_story_hours", "review_score"]].dropna().corr().iloc[0, 1]

# Platform with highest ratio
ratio_df2 = df[["main_story_hours", "completionist_hours", "platform"]].dropna()
ratio_df2 = ratio_df2[
    (ratio_df2["main_story_hours"] > 0.5)
    & (ratio_df2["completionist_hours"] > 0)
]
ratio_df2["comp_ratio"] = ratio_df2["completionist_hours"] / ratio_df2["main_story_hours"]
ratio_df2["primary_platform"] = ratio_df2["platform"].str.split(",").str[0].str.strip()
plat_ratio = (
    ratio_df2.groupby("primary_platform")["comp_ratio"]
    .agg(["median", "count"])
    .query("count >= 10")
    .sort_values("median", ascending=False)
)
top_ratio_platform = plat_ratio.index[0] if len(plat_ratio) > 0 else "N/A"
top_ratio_val = plat_ratio["median"].iloc[0] if len(plat_ratio) > 0 else 0

print(f"Total games: {total_games:,}")
print(f"Median main story: {ms_median:.1f}h, Mean: {ms_mean:.1f}h")
print(f"Overall completionist ratio: {overall_ratio:.2f}x")
print(f"Games with main story data: {has_main:,} ({pct_main:.1f}%)")
print(f"Review-playtime correlation: {corr_review:.3f}")
print(f"Top completionist platform: {top_ratio_platform} ({top_ratio_val:.2f}x)")

### Key Findings

1. **Distribution shape**: The median main story completion time across all games is significantly lower than the mean, confirming a heavily right-skewed distribution where most games are relatively short but a long tail of 100+ hour titles inflates the average — this means the median is the more useful benchmark for typical gaming experiences.

2. **Completionist multiplier**: The median completionist-to-main-story ratio shows that pursuing 100% completion roughly doubles or triples the time investment compared to the main story alone, a critical insight for players deciding whether the additional content justifies the time commitment.

3. **Platform differences**: Games available on certain platforms consistently require longer completion times than others, reflecting the different game design philosophies and content expectations across ecosystems like PC, PlayStation, and Nintendo platforms.

4. **Data coverage**: A significant proportion of games in the database have recorded main story completion times, demonstrating that the HLTB community provides broad coverage — however, the remaining entries without data tend to be niche or very old titles with small player bases.

5. **Length ≠ quality**: The correlation between review score and main story playtime is weak, indicating that game length is not a meaningful predictor of perceived quality — short games like *Portal* score as highly as 100-hour RPGs, contradicting the common assumption that longer games deliver more value per dollar.